<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/extract_gsm8k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import pandas as pd
from google.colab import files


# -------------------------
# Settings
# -------------------------

INPUT_FILE = "/content/gsm8k_rate_0.1.jsonl"

OUTPUT_JSONL = "/content/gsm8k_original_reference.jsonl"
OUTPUT_CSV = "/content/gsm8k_original_reference.csv"


# -------------------------
# Extract original data
# -------------------------

original_records = []

with open(INPUT_FILE, "r", encoding="utf-8-sig") as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)

            original_prompt = record.get(
                "original_text_backup",
                ""
            )

            gold_answer = record.get(
                "answer",
                ""
            )

            if not original_prompt:
                print(
                    f"Warning: no original_text_backup "
                    f"on line {line_number}"
                )
                continue

            original_records.append({
                "Original_Row": line_number,
                "Original_Prompt": original_prompt,
                "Gold_Answer": gold_answer
            })

        except json.JSONDecodeError as error:
            print(
                f"Invalid JSON on line {line_number}: "
                f"{error}"
            )


print(
    f"Extracted {len(original_records)} "
    f"original GSM8K records."
)


# -------------------------
# Remove exact duplicates
# -------------------------

original_df = pd.DataFrame(original_records)

before_deduplication = len(original_df)

original_df = original_df.drop_duplicates(
    subset=["Original_Prompt"],
    keep="first"
).reset_index(drop=True)

after_deduplication = len(original_df)

print(
    f"Removed "
    f"{before_deduplication - after_deduplication} "
    f"duplicate prompts."
)

print(
    f"Final original prompts: "
    f"{after_deduplication}"
)


# -------------------------
# Add an original ID
# -------------------------

original_df.insert(
    0,
    "Original_ID",
    [
        f"GSM8K_{index:04d}"
        for index in range(
            1,
            len(original_df) + 1
        )
    ]
)


# -------------------------
# Display data
# -------------------------

display(original_df.head(10))


# -------------------------
# Save CSV
# -------------------------

original_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved: {OUTPUT_CSV}")


# -------------------------
# Save JSONL
# -------------------------

with open(
    OUTPUT_JSONL,
    "w",
    encoding="utf-8"
) as file:

    for record in original_df.to_dict(
        orient="records"
    ):
        file.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )

print(f"JSONL saved: {OUTPUT_JSONL}")


# -------------------------
# Download files
# -------------------------

files.download(OUTPUT_CSV)
files.download(OUTPUT_JSONL)

Extracted 1319 original GSM8K records.
Removed 0 duplicate prompts.
Final original prompts: 1319


,Original_ID,Original_Row,Original_Prompt,Gold_Answer
0,GSM8K_0001,1,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...
1,GSM8K_0002,2,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...
2,GSM8K_0003,3,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...
3,GSM8K_0004,4,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...
4,GSM8K_0005,5,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t..."
5,GSM8K_0006,6,Kylar went to the store to buy glasses for his...,The discount price of one glass is 60/100 * 5 ...
6,GSM8K_0007,7,Toulouse has twice as many sheep as Charleston...,"If Seattle has 20 sheep, Charleston has 4 * 20..."
7,GSM8K_0008,8,Carla is downloading a 200 GB file. Normally s...,First find how many gigabytes are in 40% of th...
8,GSM8K_0009,9,John drives for 3 hours at a speed of 60 mph a...,When he turned around he was 3*60=<<3*60=180>>...
9,GSM8K_0010,10,Eliza's rate per hour for the first 40 hours s...,Eliza is entitled to 45 -40 = <<45-40=5>>5 hou...


CSV saved: /content/gsm8k_original_reference.csv
JSONL saved: /content/gsm8k_original_reference.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import os
import random
import pandas as pd
from google.colab import files


# =========================================================
# 1. Settings
# =========================================================

RANDOM_SEED = 42

# Randomly sample 10 base prompts.
# Each selected prompt retains all 8 ESL-language versions.
#
# 10 base prompts × 8 languages = 80 rows
SAMPLE_SIZE = 10

BENCHMARK_NAME = "GSM8K"

AUDIT_FILE = "/content/GSM8K_ESL_match_audit.csv"

LANGUAGES = [
    "Arabic",
    "Chinese",
    "French",
    "German",
    "Japanese",
    "Portuguese",
    "Russian",
    "Spanish"
]


# =========================================================
# 2. Check and load the matching audit
# =========================================================

if not os.path.exists(AUDIT_FILE):
    raise FileNotFoundError(
        f"File not found: {AUDIT_FILE}\n"
        "Please upload GSM8K_ESL_match_audit.csv to Colab."
    )

audit_df = pd.read_csv(
    AUDIT_FILE,
    encoding="utf-8-sig"
)

print(f"Loaded matching rows: {len(audit_df)}")


# =========================================================
# 3. Check required columns
# =========================================================

required_columns = [
    "ESL_Language",
    "Original_ID",
    "Original_Prompt",
    "Modified_Prompt",
    "Gold_Answer",
    "Match_Status"
]

missing_columns = [
    column
    for column in required_columns
    if column not in audit_df.columns
]

if missing_columns:
    raise ValueError(
        "The audit file is missing these columns: "
        f"{missing_columns}"
    )


# =========================================================
# 4. Keep only high-confidence matched records
# =========================================================

matched_df = audit_df[
    audit_df["Match_Status"] == "Auto_Matched"
].copy()

matched_df = matched_df[
    matched_df["ESL_Language"].isin(LANGUAGES)
].copy()

print(
    "Auto-matched rows for the 8 languages: "
    f"{len(matched_df)}"
)


# =========================================================
# 5. Remove accidental duplicate matches
# =========================================================
#
# If the same Original_ID-language combination appears
# more than once, keep the row with the highest similarity.
# =========================================================

if "Text_Similarity" in matched_df.columns:
    matched_df = matched_df.sort_values(
        by="Text_Similarity",
        ascending=False
    )

matched_df = matched_df.drop_duplicates(
    subset=[
        "Original_ID",
        "ESL_Language"
    ],
    keep="first"
)

print(
    "Rows after duplicate removal: "
    f"{len(matched_df)}"
)


# =========================================================
# 6. Find base prompts available in all 8 languages
# =========================================================

language_counts = (
    matched_df
    .groupby("Original_ID")["ESL_Language"]
    .nunique()
)

common_original_ids = (
    language_counts[
        language_counts == len(LANGUAGES)
    ]
    .index
    .tolist()
)

common_original_ids = sorted(common_original_ids)

print(
    "Base prompts successfully matched across all "
    f"8 languages: {len(common_original_ids)}"
)

if len(common_original_ids) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(common_original_ids)} fully matched "
        f"base prompts are available. "
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# =========================================================
# 7. Randomly sample 10 base prompts
# =========================================================
#
# Random sampling unit = base prompt.
#
# The same 10 base prompts are retained across
# all 8 ESL-language conditions.
# =========================================================

random_generator = random.Random(RANDOM_SEED)

selected_original_ids = random_generator.sample(
    common_original_ids,
    SAMPLE_SIZE
)

print(f"\nRandom seed: {RANDOM_SEED}")
print("Selected base prompts:")

for original_id in selected_original_ids:
    print(original_id)


# =========================================================
# 8. Retain all 8 language versions
# =========================================================

sample_df = matched_df[
    matched_df["Original_ID"].isin(
        selected_original_ids
    )
].copy()


# =========================================================
# 9. Create Base_ID and Sample_ID
# =========================================================

base_id_map = {
    original_id: (
        f"{BENCHMARK_NAME}_ESL_{index:03d}"
    )
    for index, original_id in enumerate(
        selected_original_ids,
        start=1
    )
}

sample_df["Base_ID"] = (
    sample_df["Original_ID"]
    .map(base_id_map)
)

sample_df["Sample_ID"] = (
    sample_df["Base_ID"]
    + "_"
    + sample_df["ESL_Language"]
)

sample_df["Benchmark"] = BENCHMARK_NAME


# GSM8K has no answer choices
if "Answer_Choices" not in sample_df.columns:
    sample_df["Answer_Choices"] = ""


# =========================================================
# 10. Structural checks
# =========================================================

language_count_per_prompt = (
    sample_df
    .groupby("Original_ID")["ESL_Language"]
    .nunique()
)

assert (
    sample_df["Original_ID"].nunique()
    == SAMPLE_SIZE
), (
    "The sample does not contain exactly "
    f"{SAMPLE_SIZE} base prompts."
)

assert (
    language_count_per_prompt == len(LANGUAGES)
).all(), (
    "At least one selected base prompt does not "
    "have all 8 ESL-language versions."
)

assert len(sample_df) == (
    SAMPLE_SIZE * len(LANGUAGES)
), (
    f"Expected {SAMPLE_SIZE * len(LANGUAGES)} rows, "
    f"but found {len(sample_df)}."
)

assert sample_df["Sample_ID"].is_unique, (
    "Sample_ID values are not unique."
)


# =========================================================
# 11. Confirm balanced language representation
# =========================================================

language_summary = (
    sample_df["ESL_Language"]
    .value_counts()
    .reindex(LANGUAGES)
)

print("\nNumber of rows per ESL-language condition:")
print(language_summary)

assert (
    language_summary == SAMPLE_SIZE
).all(), (
    "At least one language does not have exactly "
    f"{SAMPLE_SIZE} samples."
)

print("\nAll structural checks passed.")
print(
    "Unique base prompts:",
    sample_df["Original_ID"].nunique()
)
print(
    "Final ESL validation rows:",
    len(sample_df)
)


# =========================================================
# 12. Add reviewer columns
# =========================================================

sample_df["R1_Meaning"] = ""
sample_df["R2_Meaning"] = ""
sample_df["Final_Meaning"] = ""

sample_df["R1_Key_Info"] = ""
sample_df["R2_Key_Info"] = ""
sample_df["Final_Key_Info"] = ""

sample_df["R1_ESL_Plausibility"] = ""
sample_df["R2_ESL_Plausibility"] = ""
sample_df["Final_ESL_Plausibility"] = ""

sample_df["R1_Readability"] = ""
sample_df["R2_Readability"] = ""
sample_df["Final_Readability"] = ""

sample_df["R1_Comments"] = ""
sample_df["R2_Comments"] = ""
sample_df["Adjudication_Comments"] = ""


# =========================================================
# 13. Organize Master columns
# =========================================================

tracking_columns = [
    column
    for column in [
        "Original_Row",
        "ESL_Source_Row",
        "Text_Similarity",
        "Number_Overlap",
        "Match_Status"
    ]
    if column in sample_df.columns
]

master_columns = [
    "Base_ID",
    "Sample_ID",
    "Benchmark",
    "ESL_Language",
    "Original_ID"
] + tracking_columns + [
    "Original_Prompt",
    "Modified_Prompt",
    "Answer_Choices",
    "Gold_Answer",

    "R1_Meaning",
    "R2_Meaning",
    "Final_Meaning",

    "R1_Key_Info",
    "R2_Key_Info",
    "Final_Key_Info",

    "R1_ESL_Plausibility",
    "R2_ESL_Plausibility",
    "Final_ESL_Plausibility",

    "R1_Readability",
    "R2_Readability",
    "Final_Readability",

    "R1_Comments",
    "R2_Comments",
    "Adjudication_Comments"
]

sample_df = sample_df[
    master_columns
]


# =========================================================
# 14. Shuffle the 80 annotation rows
# =========================================================
#
# This prevents reviewers from seeing all 8 language
# versions of the same base prompt consecutively.
# =========================================================

sample_df = sample_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

sample_df.insert(
    0,
    "Annotation_Order",
    range(1, len(sample_df) + 1)
)


# =========================================================
# 15. Save Master file
# =========================================================

master_file = (
    "/content/"
    "GSM8K_ESL_balanced_matched_"
    "sample_n10_seed42_MASTER.csv"
)

sample_df.to_csv(
    master_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nMaster file saved: {master_file}")


# =========================================================
# 16. Create Reviewer 1 file
# =========================================================
#
# Hidden from Reviewer 1:
# - ESL_Language
# - Original_ID
# - Source rows
# - Gold_Answer
# - Similarity scores
# - R2 and Final columns
# =========================================================

reviewer_1_columns = [
    "Sample_ID",
    "Benchmark",
    "Original_Prompt",
    "Modified_Prompt",
    "Answer_Choices",

    "R1_Meaning",
    "R1_Key_Info",
    "R1_ESL_Plausibility",
    "R1_Readability",
    "R1_Comments"
]

reviewer_1_df = sample_df[
    reviewer_1_columns
].copy()

reviewer_1_df = reviewer_1_df.sample(
    frac=1,
    random_state=101
).reset_index(drop=True)

reviewer_1_df.insert(
    0,
    "Annotation_Order",
    range(1, len(reviewer_1_df) + 1)
)

reviewer_1_file = (
    "/content/"
    "GSM8K_ESL_balanced_matched_"
    "sample_n10_seed42_R1.csv"
)

reviewer_1_df.to_csv(
    reviewer_1_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Reviewer 1 file saved: {reviewer_1_file}")


# =========================================================
# 17. Create Reviewer 2 file
# =========================================================
#
# Reviewer 2 receives a different random row order.
# Sample_ID remains unchanged for later merging.
# =========================================================

reviewer_2_columns = [
    "Sample_ID",
    "Benchmark",
    "Original_Prompt",
    "Modified_Prompt",
    "Answer_Choices",

    "R2_Meaning",
    "R2_Key_Info",
    "R2_ESL_Plausibility",
    "R2_Readability",
    "R2_Comments"
]

reviewer_2_df = sample_df[
    reviewer_2_columns
].copy()

reviewer_2_df = reviewer_2_df.sample(
    frac=1,
    random_state=202
).reset_index(drop=True)

reviewer_2_df.insert(
    0,
    "Annotation_Order",
    range(1, len(reviewer_2_df) + 1)
)

reviewer_2_file = (
    "/content/"
    "GSM8K_ESL_balanced_matched_"
    "sample_n10_seed42_R2.csv"
)

reviewer_2_df.to_csv(
    reviewer_2_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Reviewer 2 file saved: {reviewer_2_file}")


# =========================================================
# 18. Save selected base-prompt IDs
# =========================================================

selected_ids_df = pd.DataFrame({
    "Selection_Order": range(
        1,
        SAMPLE_SIZE + 1
    ),
    "Benchmark": BENCHMARK_NAME,
    "Original_ID": selected_original_ids,
    "Random_Seed": RANDOM_SEED
})

selected_ids_file = (
    "/content/"
    "GSM8K_ESL_selected_base_prompt_ids_"
    "n10_seed42.csv"
)

selected_ids_df.to_csv(
    selected_ids_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Selected prompt IDs saved: "
    f"{selected_ids_file}"
)


# =========================================================
# 19. Save sampling summary
# =========================================================

summary_df = pd.DataFrame([
    {
        "Benchmark": BENCHMARK_NAME,

        "Available_Fully_Matched_Base_Prompts": (
            len(common_original_ids)
        ),

        "Selected_Base_Prompts": SAMPLE_SIZE,

        "ESL_Language_Conditions": (
            len(LANGUAGES)
        ),

        "Samples_Per_Language": SAMPLE_SIZE,

        "Final_Annotation_Rows": len(sample_df),

        "Random_Seed": RANDOM_SEED,

        "Sampling_Design": (
            "Balanced matched random sampling "
            "across ESL language conditions"
        )
    }
])

summary_file = (
    "/content/"
    "GSM8K_ESL_sampling_summary_"
    "n10_seed42.csv"
)

summary_df.to_csv(
    summary_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Sampling summary saved: {summary_file}")


# =========================================================
# 20. Final checks
# =========================================================

assert len(sample_df) == 80
assert sample_df["Base_ID"].nunique() == 10
assert sample_df["Sample_ID"].nunique() == 80
assert sample_df["Sample_ID"].is_unique

print("\nFinal checks passed.")
print("Expected base prompts: 10")
print(
    "Actual base prompts:",
    sample_df["Base_ID"].nunique()
)
print("Expected rows: 80")
print("Actual rows:", len(sample_df))


# =========================================================
# 21. Display results
# =========================================================

display(summary_df)

display(
    language_summary
    .rename("Sample_Count")
    .reset_index()
    .rename(columns={
        "index": "ESL_Language"
    })
)

display(sample_df.head(10))


# =========================================================
# 22. Download output files
# =========================================================

files.download(master_file)
files.download(reviewer_1_file)
files.download(reviewer_2_file)
files.download(selected_ids_file)
files.download(summary_file)

Loaded matching rows: 9748
Auto-matched rows for the 8 languages: 9270
Rows after duplicate removal: 9270
Base prompts successfully matched across all 8 languages: 1013

Random seed: 42
Selected base prompts:
GSM8K_0862
GSM8K_0153
GSM8K_0032
GSM8K_0991
GSM8K_0362
GSM8K_0323
GSM8K_0297
GSM8K_0187
GSM8K_0985
GSM8K_0138

Number of rows per ESL-language condition:
ESL_Language
Arabic        10
Chinese       10
French        10
German        10
Japanese      10
Portuguese    10
Russian       10
Spanish       10
Name: count, dtype: int64

All structural checks passed.
Unique base prompts: 10
Final ESL validation rows: 80

Master file saved: /content/GSM8K_ESL_balanced_matched_sample_n10_seed42_MASTER.csv
Reviewer 1 file saved: /content/GSM8K_ESL_balanced_matched_sample_n10_seed42_R1.csv
Reviewer 2 file saved: /content/GSM8K_ESL_balanced_matched_sample_n10_seed42_R2.csv
Selected prompt IDs saved: /content/GSM8K_ESL_selected_base_prompt_ids_n10_seed42.csv
Sampling summary saved: /content/GSM8K

,Benchmark,Available_Fully_Matched_Base_Prompts,Selected_Base_Prompts,ESL_Language_Conditions,Samples_Per_Language,Final_Annotation_Rows,Random_Seed,Sampling_Design
0,GSM8K,1013,10,8,10,80,42,Balanced matched random sampling across ESL la...


,ESL_Language,Sample_Count
0,Arabic,10
1,Chinese,10
2,French,10
3,German,10
4,Japanese,10
5,Portuguese,10
6,Russian,10
7,Spanish,10


,Annotation_Order,Base_ID,Sample_ID,Benchmark,ESL_Language,Original_ID,Original_Row,ESL_Source_Row,Text_Similarity,Number_Overlap,...,Final_Key_Info,R1_ESL_Plausibility,R2_ESL_Plausibility,Final_ESL_Plausibility,R1_Readability,R2_Readability,Final_Readability,R1_Comments,R2_Comments,Adjudication_Comments
0,1,GSM8K_ESL_008,GSM8K_ESL_008_Arabic,GSM8K,Arabic,GSM8K_0187,187,169,0.9562,1.0,...,,,,,,,,,,
1,2,GSM8K_ESL_003,GSM8K_ESL_003_Arabic,GSM8K,Arabic,GSM8K_0032,32,28,1.0000,1.0,...,,,,,,,,,,
2,3,GSM8K_ESL_008,GSM8K_ESL_008_German,GSM8K,German,GSM8K_0187,187,169,0.9663,1.0,...,,,,,,,,,,
3,4,GSM8K_ESL_007,GSM8K_ESL_007_Chinese,GSM8K,Chinese,GSM8K_0297,297,271,0.9553,1.0,...,,,,,,,,,,
4,5,GSM8K_ESL_009,GSM8K_ESL_009_Spanish,GSM8K,Spanish,GSM8K_0985,985,907,0.9694,1.0,...,,,,,,,,,,
5,6,GSM8K_ESL_008,GSM8K_ESL_008_French,GSM8K,French,GSM8K_0187,187,169,0.9591,1.0,...,,,,,,,,,,
6,7,GSM8K_ESL_003,GSM8K_ESL_003_Spanish,GSM8K,Spanish,GSM8K_0032,32,28,0.9903,1.0,...,,,,,,,,,,
7,8,GSM8K_ESL_001,GSM8K_ESL_001_Spanish,GSM8K,Spanish,GSM8K_0862,862,791,0.7733,1.0,...,,,,,,,,,,
8,9,GSM8K_ESL_003,GSM8K_ESL_003_French,GSM8K,French,GSM8K_0032,32,28,1.0000,1.0,...,,,,,,,,,,
9,10,GSM8K_ESL_004,GSM8K_ESL_004_Spanish,GSM8K,Spanish,GSM8K_0991,991,913,0.9840,1.0,...,,,,,,,,,,


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>